# Effective redshift and Effective Volume:

In [5]:
import os
import sys
import logging

import numpy as np
from matplotlib import pyplot as plt

from scipy.interpolate import interp1d

import pandas as pd
import healpy as hp
from regressis.utils import read_fits_to_pandas
from regressis.footprint import DR9Footprint, DESIFootprint

# Define Footprint:
nside = 256
footprint = DESIFootprint(nside)
dr9_footprint = DR9Footprint(nside, mask_around_des=False, cut_desi=False)

# Calcul redshift effectif (comme dans Castorina ou Arnaud):
from cosmoprimo.fiducial import DESI
cosmo = DESI()

%matplotlib inline

In [13]:
nbr_randoms = 2

def load_data(tracer, nbr_randoms=nbr_randoms, nside=nside, LSS='/global/cfs/cdirs/desi/survey/catalogs/Y3/LSS/loa-v1/LSScats/v2/nonKP'):
    regions = ['NGC', 'SGC']
    randoms_fn, data_fn = '{}_{}_{}_clustering.ran.fits', '{}_{}_clustering.dat.fits'
    
    data = pd.concat([read_fits_to_pandas(os.path.join(LSS, data_fn.format(tracer, region)), columns=['RA', 'DEC', 'Z', 'WEIGHT', 'WEIGHT_COMP', 'WEIGHT_FKP', 'NX']) for region in regions])
    
    randoms = pd.concat([pd.concat([read_fits_to_pandas(os.path.join(LSS, randoms_fn.format(tracer, 'NGC', i)), columns=['RA', 'DEC', 'Z', 'WEIGHT_FKP', 'FRAC_TLOBS_TILES']) for i in range(nbr_randoms)]), 
                         pd.concat([read_fits_to_pandas(os.path.join(LSS, randoms_fn.format(tracer, 'SGC', i)), columns=['RA', 'DEC', 'Z', 'WEIGHT_FKP', 'FRAC_TLOBS_TILES']) for i in range(nbr_randoms)])])   
    
    data['HPX'] = hp.ang2pix(nside, data['RA'].values, data['DEC'].values, nest=True, lonlat=True)
    randoms['HPX'] = hp.ang2pix(nside, randoms['RA'].values, randoms['DEC'].values, nest=True, lonlat=True)

    return data, randoms

def compute_nz(data, data_weights, randoms, randoms_weights, zmin, zmax, nbins=51, cosmo=cosmo, nbr_randoms=nbr_randoms, return_bin_vol=False):
        """
        randoms_weights=randoms['FRAC_TLOBS_TILES'] -> use effective area to normalized the n(z). WHAT WE NEED TO USE IF WEIGHT_COMP IS USED in the data ! (it contains the rest of the completeness weights.)
        randoms_weights=np.ones_like(...) -> gives the real geometric area. Useful to compute the real n(z) without any correction.
        Note: area is useless for effective redshift compution since we normalize by nz**2 as well.

        # add the last bit of completeness weights into the effective area of the survey:   
        area = randoms['FRAC_TLOBS_TILES'].sum() / (nbr_randoms * 2500)
        # This is the real geometrical area size, what we actually observe:
        area = randoms['FRAC_TLOBS_TILES'].size / (nbr_randoms * 2500)        
        """
        # generate n(z)
        nz, zedges = np.histogram(data['Z'], weights=data_weights, bins=nbins, range=(zmin, zmax))
        z = (zedges[1:] + zedges[:-1]) / 2

        area = randoms_weights.sum() / (nbr_randoms * 2500)
    
        # Volume computation for each redshift bins:
        vol = area / (360*360/np.pi) *4/3*np.pi * (cosmo.comoving_radial_distance(zedges[1:])**3 - cosmo.comoving_radial_distance(zedges[:-1])**3)  

        # Get the n(z) in (Mpc/h)**-3:
        nz =  nz / vol
        
        if return_bin_vol:
            return z, nz, vol
        return z, nz


def dchi_dz(z, cosmo):
    """
    Convert a comoving distance interval dr to a redshift interval dz. (dz = dr * H(z) / c) 
    Warning: take care of the units ! We want dchi/dz in Mpc/h, so we need to use H0 in m.s^-1.h.Mpc^-1 and c in m/s.
    """
    from cosmoprimo import constants
    H0 = 100 * 1000  # en m.s^-1.h.Mpc^-1
    return constants.c / (H0 * cosmo.get_background().efunc(z))


def effective_volume(data, data_weights, randoms, randoms_weights, P0, zmin, zmax,  
                     data2=None, data_weights2=None, randoms2=None, randoms_weights2=None, P02=None, nbins=51, cosmo=cosmo):
        """ P0 can be either constant or an np.array. """
        z, nz, vol = compute_nz(data, data_weights, randoms, randoms_weights, zmin, zmax, cosmo=cosmo, nbins=nbins, return_bin_vol=True)
        if data2 is not None: 
            z2, nz2, vol2 = compute_nz(data2, data_weights2, randoms2, randoms_weights2, zmin, zmax, cosmo=cosmo, nbins=nbins, return_bin_vol=True)
        else:
            z2, nz2, vol2, P02 = z, nz, vol, P0

        return np.sum((np.sqrt(vol) * (nz*P0) / (1+nz*P0)) * (np.sqrt(vol2) * (nz2*P02) / (1+nz2*P02))) 


def effective_redshift(data, data_weights, randoms, randoms_weights, zmin, zmax, 
                       data2=None, data_weights2=None, randoms2=None, randoms_weights2=None, cosmo=cosmo):
    """
    Compute effective redshift (Appendix B of https://arxiv.org/pdf/2007.09008)
    
    Warning: the integration is over dr^3 -> can be simplied to dr/dz * r^2 * dz (angular part is not important since it is the same in the numerator and denominator).
    """
    # compute n(z):
    z, nz = compute_nz(data, data_weights, randoms, randoms_weights, zmin, zmax, cosmo=cosmo)
    if data2 is not None:
        _, nz2 = compute_nz(data2, data_weights2, randoms2, randoms_weights2, zmin, zmax, cosmo=cosmo)
    else:
        nz2 = nz

    # remark: this vol that I can return from compute_nz(return_vol_bin=True)
    r2_dr_dz = cosmo.comoving_radial_distance(z)**2 * dchi_dz(z, cosmo)

    # how I compute OQE weight (byb taking sqrt can lead to negative value, check that it is very few ... at very low-z)
    return np.nansum(nz * nz2 * z * r2_dr_dz) / np.nansum(nz * nz2 * r2_dr_dz)

In [ ]:
def bias(z, tracer='QSO', return_coeffs=False):
    """
    Bias model for the different DESI tracer (measured from DR2 data).
    """

    params = {'BGS_BRIGHT-21.35': (0.60646037, 0.60646037),
              'LRG': (0.23553567, 1.3458994),
              'ELG_LOPnotqso': (0.15066781, 0.59463735),
              'ELGnotqso': (0.15487521, 0.59464828),
              'QSO': (0.25207547, 0.71020952)}

    if tracer in params:
        alpha, beta = params[tracer]
    else:
        print(f'Tracer: {tracer} is not ready...')
        
    if return_coeffs:
        return alpha, beta
    else:
        return alpha*(1+z)**2 + beta

def w_fkp_nx(data, P0): return 1 / (1 + data['NX']*P0)

# take care here nz need to be computed without the completeness ?
def w_fkp(z, P0, nz): return 1 / (1 + nz(z)*P0) 

def w_tilde(z, tracer, pop=1.6): return (bias(z, tracer=tracer) - pop)
def w0(z,tracer): 
    z = np.asarray(z, dtype=np.float64).copy() # ← ensures writable, contiguous float64
    return cosmo.growth_factor(z) * (bias(z, tracer=tracer) + cosmo.growth_rate(z) / 3)
def w2(z): 
    z = np.asarray(z, dtype=np.float64).copy() # ← ensures writable, contiguous float64
    return 2 / 3 * cosmo.growth_factor(z) * cosmo.growth_rate(z)

In [24]:
def display_effective_redshift(tracer, zmin, zmax, P0=5e4, p=1.0):

    LSS_dr2 = '/global/cfs/cdirs/desi/survey/catalogs/Y3/LSS/loa-v1/LSScats/v2/nonKP'

    dir_fn = '/global/cfs/cdirs/desi/science/cai/desi-clustering/dr2/summary_statistics/local_png/base/desi-data/loa-v1/v2/fNL/blinded/'

    data, randoms = load_data(tracer, LSS=LSS_dr2)

    print(f"\n{tracer}: {zmin} < z < {zmax}; Veff={effective_volume(data, data['WEIGHT'], randoms, randoms['FRAC_TLOBS_TILES'], zmin, zmax, P0)/1e9:2.3f} [Gpc/h]**3") 
    
    # Randoms weights are useless for effective redshift computation since area**2 is normalized.
    print(f"zeff / b1(zeff) (only w.):     {effective_redshift(data, data['WEIGHT'], randoms, np.ones_like(randoms['Z']), zmin, zmax):2.3f} / {bias(effective_redshift(data, data['WEIGHT'], randoms, np.ones_like(randoms['Z']), zmin, zmax), tracer):2.3f}")
    print(f"zeff / b1(zeff) (BAO FKP):     {effective_redshift(data, data['WEIGHT']*data['WEIGHT_FKP'], randoms, np.ones_like(randoms['Z']), zmin, zmax):2.3f} / {bias(effective_redshift(data, data['WEIGHT']*data['WEIGHT_FKP'], randoms, np.ones_like(randoms['Z']), zmin, zmax), tracer):2.3f}")
    print(f"zeff / b1(zeff) (fNL FKP):     {effective_redshift(data, data['WEIGHT']*w_fkp_nx(data, P0), randoms, np.ones_like(randoms['Z']), zmin, zmax):2.3f} / {bias(effective_redshift(data, data['WEIGHT']*w_fkp_nx(data, 5e4), randoms, np.ones_like(randoms['Z']), zmin, zmax), tracer):2.3f}")
    print(f"zeff / b1(zeff) (OQE (ell=0)): {effective_redshift(data, data['WEIGHT']*w_fkp_nx(data, P0)*np.sqrt(w_tilde(data['Z'], tracer, pop=p)*w0(data['Z'], tracer)), randoms, np.ones_like(randoms['Z']), zmin, zmax):2.3f} / {bias(effective_redshift(data, data['WEIGHT']*w_fkp_nx(data, 5e4)*np.sqrt(w_tilde(data['Z'], tracer, pop=p)*w0(data['Z'], tracer)), randoms, np.ones_like(randoms['Z']), zmin, zmax), tracer):2.3f}")
    print(f"zeff / b1(zeff) (OQE (ell=2)): {effective_redshift(data, data['WEIGHT']*w_fkp_nx(data, P0)*np.sqrt(w_tilde(data['Z'], tracer, pop=p)*w2(data['Z'])), randoms, np.ones_like(randoms['Z']), zmin, zmax):2.3f} / {bias(effective_redshift(data, data['WEIGHT']*w_fkp_nx(data, 5e4)*np.sqrt(w_tilde(data['Z'], tracer, pop=p)*w2(data['Z'])), randoms, np.ones_like(randoms['Z']), zmin, zmax), tracer):2.3f}")

    try:
        import lsstypes
        for weight in ['default-fkp', 'default-fkp-oqe']:
            fn = f'window_mesh2_spectrum_poles_{tracer}_z{zmin}-{zmax}_GCcomb_weight-{weight}.h5'
            w = lsstypes.read(dir_fn + fn)
            print(f"From desi-clustering (stored in the window matrix) with {weight=}:")
            for ell in [0,2]:
                zeff = float(w.observable.get(ell).attrs['zeff'])
                print(f"{tracer=}, {zmin=}, {zmax=}, {ell=}, {zeff=:2.2f}")
    except:
        pass

In [ ]:
tracer = 'LRG'
P0 = 5e4
zmin, zmax = 0.4, 1.1
p = 1.0

display_effective_redshift(tracer, zmin, zmax, P0, p)


LRG: 0.4 < z < 1.1; Veff=0.000 [Gpc/h]**3
zeff / b1(zeff) (only w.):     0.711 / 2.036
zeff / b1(zeff) (BAO FKP):     0.762 / 2.077
zeff / b1(zeff) (fNL FKP):     0.794 / 2.104
zeff / b1(zeff) (OQE (ell=0)): 0.817 / 2.123
zeff / b1(zeff) (OQE (ell=2)): 0.813 / 2.120
From desi-clustering (stored in the window matrix) with weight='default-fkp':
tracer='LRG', zmin=0.4, zmax=1.1, ell=0, zeff=0.7944370294619002
tracer='LRG', zmin=0.4, zmax=1.1, ell=2, zeff=0.7944370294619002
From desi-clustering (stored in the window matrix) with weight='default-fkp-oqe':
tracer='LRG', zmin=0.4, zmax=1.1, ell=0, zeff=0.81777244435554
tracer='LRG', zmin=0.4, zmax=1.1, ell=2, zeff=0.8140168882195823


In [25]:
tracer = 'QSO'
P0 = 3e4
zmin, zmax = 0.8, 3.5
p = 1.4

display_effective_redshift(tracer, zmin, zmax, P0, p)


QSO: 0.8 < z < 3.5; Veff=0.000 [Gpc/h]**3
zeff / b1(zeff) (only w.):     1.631 / 2.455
zeff / b1(zeff) (BAO FKP):     1.657 / 2.489
zeff / b1(zeff) (fNL FKP):     1.734 / 2.658


/global/common/software/desi/users/adematti/perlmutter/cosmodesiconda/20260321-1.0.0/conda/lib/python3.12/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


zeff / b1(zeff) (OQE (ell=0)): 2.104 / 3.246
zeff / b1(zeff) (OQE (ell=2)): 1.987 / 3.050
From desi-clustering (stored in the window matrix) with weight='default-fkp':
tracer='QSO', zmin=0.8, zmax=3.5, ell=0, zeff=1.74
tracer='QSO', zmin=0.8, zmax=3.5, ell=2, zeff=1.74
From desi-clustering (stored in the window matrix) with weight='default-fkp-oqe':
tracer='QSO', zmin=0.8, zmax=3.5, ell=0, zeff=2.13
tracer='QSO', zmin=0.8, zmax=3.5, ell=2, zeff=2.00
